# Fine-tune PhoBERT-base-v2 -- Phân loại cảm xúc tiếng Việt (3 lớp)

## Mục tiêu
Fine-tune mô hình `vinai/phobert-base-v2` cho bài toán phân loại cảm xúc 3 lớp:
- **0 = neg** (Tiêu cực)
- **1 = pos** (Tích cực)
- **2 = neu** (Trung tính)

## Dataset
- **File**: `/content/drive/MyDrive/data/data_v2/data_v2.csv`
- **Cột**: `Review`, `Label_number`, `Label_text`

## Pipeline
1. Mount Google Drive và load dữ liệu
2. Kiểm tra cấu trúc, làm sạch dữ liệu
3. Phân tích dữ liệu (EDA)
4. Tokenization với PhoBERT tokenizer
5. Chia train/validation
6. Fine-tune với HuggingFace Trainer
7. Đánh giá mô hình (Accuracy, F1, Confusion Matrix)
8. Lưu mô hình và inference demo

## Tối ưu cho GPU T4 (Google Colab)
- Mixed precision (fp16)
- Gradient accumulation
- Dynamic padding
- Early stopping

In [ ]:
# =============================================================================
# Cell 1: Cài đặt thư viện cần thiết
# =============================================================================
# Phiên bản transformers tương thích với PhoBERT-base-v2
!pip install -q transformers==4.44.2 datasets accelerate scikit-learn seaborn matplotlib

In [ ]:
# =============================================================================
# Cell 2: Import thư viện và cấu hình môi trường
# =============================================================================
import os
import sys
import random
import warnings
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# -- Đặt seed để đảm bảo kết quả lặp lại được --
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# -- Kiểm tra GPU --
if not torch.cuda.is_available():
    print("[CẢNH BÁO] Không tìm thấy GPU.")
    print("Vào Runtime -> Change runtime type -> chọn GPU (T4).")
    sys.exit(1)

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU    : {gpu_name} ({gpu_mem:.1f} GB)")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

# -- Cấu hình chung --
MODEL_NAME = "vinai/phobert-base-v2"
NUM_LABELS = 3
MAX_LENGTH = 256

LABEL_MAP = {0: "neg", 1: "pos", 2: "neu"}
LABEL_NAMES = ["neg", "pos", "neu"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

In [ ]:
# =============================================================================
# Cell 3: Mount Google Drive và load dataset
# =============================================================================
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# -- Đường dẫn dữ liệu và thư mục lưu mô hình --
DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_ROOT = DRIVE_ROOT / "DO_AN_1_main"
OUTPUT_DIR = str(PROJECT_ROOT / "phobert-v2")
DATA_FILENAME = "data_v2.csv"

# Tạo thư mục lưu mô hình nếu chưa tồn tại
os.makedirs(OUTPUT_DIR, exist_ok=True)


def resolve_data_path() -> str:
    """Tự động tìm file dữ liệu trong Google Drive.

    Ưu tiên các đường dẫn phổ biến trước. Nếu không tìm thấy,
    sẽ quét trong MyDrive để tìm file data_v2.csv.
    """
    candidate_paths = [
        DRIVE_ROOT / "data" / "data_v2" / DATA_FILENAME,
        DRIVE_ROOT / "data_v2" / DATA_FILENAME,
        DRIVE_ROOT / "DO_AN_1" / "data" / "data_v2" / DATA_FILENAME,
        DRIVE_ROOT / "datasets" / "data_v2" / DATA_FILENAME,
        PROJECT_ROOT / "data" / "data_v2" / DATA_FILENAME,
    ]

    for candidate in candidate_paths:
        if candidate.exists():
            return str(candidate)

    matches = sorted(DRIVE_ROOT.rglob(DATA_FILENAME))
    if matches:
        print("[THÔNG BÁO] Không tìm thấy file ở path mặc định.")
        print(f"[THÔNG BÁO] Đã tự động tìm thấy file tại: {matches[0]}")
        if len(matches) > 1:
            print("[THÔNG BÁO] Có nhiều file trùng tên, notebook sẽ dùng file đầu tiên tìm thấy.")
            for idx, match in enumerate(matches[:10], start=1):
                print(f"  {idx}. {match}")
        return str(matches[0])

    raise FileNotFoundError(
        "Không tìm thấy file data_v2.csv trong Google Drive. "
        "Hãy kiểm tra lại tên file hoặc upload file vào MyDrive."
    )


DATA_PATH = resolve_data_path()
print(f"Đường dẫn dữ liệu: {DATA_PATH}")
print(f"Đường dẫn lưu model: {OUTPUT_DIR}")

# -- Đọc dữ liệu --
df = pd.read_csv(DATA_PATH)
print(f"Số lượng mẫu: {len(df)}")
print(f"Các cột     : {list(df.columns)}")
df.head()

In [ ]:
# =============================================================================
# Cell 4: Kiểm tra cấu trúc dữ liệu và làm sạch
# =============================================================================
print("=== THÔNG TIN DỮ LIỆU BAN ĐẦU ===")
print(f"Kích thước: {df.shape}")
print(f"\nKiểu dữ liệu:\n{df.dtypes}")
print(f"\nGiá trị null:\n{df.isnull().sum()}")
print(f"\nSố mẫu trùng lặp: {df.duplicated().sum()}")

# -- Làm sạch dữ liệu --
# Loại bỏ dòng có giá trị null ở cột Review hoặc Label_number
df = df.dropna(subset=["Review", "Label_number"])

# Loại bỏ dòng trùng lặp
df = df.drop_duplicates(subset=["Review"], keep="first")

# Chuyển Label_number sang kiểu int
df["Label_number"] = df["Label_number"].astype(int)

# Kiểm tra chỉ có 3 nhãn hợp lệ: 0, 1, 2
valid_labels = {0, 1, 2}
invalid_mask = ~df["Label_number"].isin(valid_labels)
if invalid_mask.any():
    print(f"\n[CẢNH BÁO] Loại bỏ {invalid_mask.sum()} dòng có nhãn không hợp lệ.")
    df = df[~invalid_mask]

# -- Làm sạch text cơ bản --
# Loại bỏ khoảng trắng thừa, dòng có review rỗng
df["Review"] = df["Review"].astype(str).str.strip()
df = df[df["Review"].str.len() > 0]

# Reset index
df = df.reset_index(drop=True)

print(f"\n=== SAU KHI LÀM SẠCH ===")
print(f"Số lượng mẫu: {len(df)}")
print(f"Phân bố nhãn:\n{df['Label_number'].value_counts().sort_index()}")

In [ ]:
# =============================================================================
# Cell 5: Phân tích dữ liệu (EDA) -- Phân bố nhãn
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# -- Biểu đồ 1: Phân bố nhãn (bar chart) --
label_counts = df["Label_number"].value_counts().sort_index()
colors = ["#e74c3c", "#2ecc71", "#3498db"]  # đỏ, xanh lá, xanh dương
bar_labels = [LABEL_MAP[i] for i in label_counts.index]

bars = axes[0].bar(bar_labels, label_counts.values, color=colors, edgecolor="black")
axes[0].set_title("Phân bố nhãn cảm xúc", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Nhãn")
axes[0].set_ylabel("Số lượng")
for bar, count in zip(bars, label_counts.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
        str(count), ha="center", fontsize=11, fontweight="bold"
    )

# -- Biểu đồ 2: Tỉ lệ nhãn (pie chart) --
axes[1].pie(
    label_counts.values,
    labels=bar_labels,
    colors=colors,
    autopct="%1.1f%%",
    startangle=90,
    textprops={"fontsize": 12},
)
axes[1].set_title("Tỉ lệ nhãn cảm xúc", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

# In tỉ lệ chi tiết
for idx, name in LABEL_MAP.items():
    count = label_counts.get(idx, 0)
    pct = count / len(df) * 100
    print(f"  {name}: {count} mẫu ({pct:.1f}%)")

In [ ]:
# =============================================================================
# Cell 6: Phân tích dữ liệu (EDA) -- Độ dài câu
# =============================================================================
# Tính độ dài theo số từ (tách bởi khoảng trắng)
df["word_count"] = df["Review"].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# -- Biểu đồ 1: Phân bố độ dài câu (histogram) --
axes[0].hist(df["word_count"], bins=50, color="#3498db", edgecolor="black", alpha=0.7)
axes[0].axvline(df["word_count"].mean(), color="red", linestyle="--", label=f"Trung bình: {df['word_count'].mean():.1f}")
axes[0].axvline(df["word_count"].median(), color="green", linestyle="--", label=f"Trung vị: {df['word_count'].median():.1f}")
axes[0].set_title("Phân bố độ dài câu (số từ)", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Số từ")
axes[0].set_ylabel("Số lượng")
axes[0].legend()

# -- Biểu đồ 2: Boxplot độ dài theo nhãn --
data_by_label = [df[df["Label_number"] == i]["word_count"].values for i in sorted(df["Label_number"].unique())]
bp = axes[1].boxplot(
    data_by_label,
    labels=[LABEL_MAP[i] for i in sorted(df["Label_number"].unique())],
    patch_artist=True,
)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title("Độ dài câu theo nhãn cảm xúc", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Nhãn")
axes[1].set_ylabel("Số từ")

plt.tight_layout()
plt.show()

# Thống kê độ dài
print("=== THỐNG KÊ ĐỘ DÀI CÂU ===")
print(f"  Trung bình : {df['word_count'].mean():.1f} từ")
print(f"  Trung vị   : {df['word_count'].median():.1f} từ")
print(f"  Nhỏ nhất   : {df['word_count'].min()} từ")
print(f"  Lớn nhất   : {df['word_count'].max()} từ")
print(f"  Độ lệch chuẩn: {df['word_count'].std():.1f} từ")

# Xóa cột tạm để không ảnh hưởng đến bước sau
df = df.drop(columns=["word_count"])

In [ ]:
# =============================================================================
# Cell 7: Tokenization với PhoBERT tokenizer
# =============================================================================
print("Đang tải PhoBERT tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer: {type(tokenizer).__name__}")
print(f"Vocab size: {tokenizer.vocab_size}")

# -- Hàm tokenize --
def tokenize_function(examples):
    """Tokenize văn bản sử dụng PhoBERT tokenizer.
    Truncate tại MAX_LENGTH, không pad ở đây (sẽ dùng dynamic padding sau).
    """
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        # Không pad ở đây -- DataCollatorWithPadding sẽ pad động theo batch
    )

# Kiểm tra thử với 1 mẫu
sample_text = df["Review"].iloc[0]
sample_tokens = tokenizer(sample_text, truncation=True, max_length=MAX_LENGTH)
print(f"\nVí dụ tokenization:")
print(f"  Text   : {sample_text[:80]}...")
print(f"  Số token: {len(sample_tokens['input_ids'])}")

In [ ]:
# =============================================================================
# Cell 8: Chia train/validation và tạo HuggingFace Dataset
# =============================================================================
# -- Chia dữ liệu: 85% train, 15% validation --
# Stratify theo nhãn để đảm bảo tỉ lệ nhãn đồng đều giữa 2 tập
train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    random_state=SEED,
    stratify=df["Label_number"],
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Tập train     : {len(train_df)} mẫu")
print(f"Tập validation: {len(val_df)} mẫu")
print(f"\nPhân bố nhãn tập train:\n{train_df['Label_number'].value_counts().sort_index()}")
print(f"\nPhân bố nhãn tập validation:\n{val_df['Label_number'].value_counts().sort_index()}")

# -- Chuyển sang HuggingFace Dataset --
# Đổi tên cột cho phù hợp: Review -> text, Label_number -> label
train_dataset = Dataset.from_pandas(
    train_df[["Review", "Label_number"]].rename(
        columns={"Review": "text", "Label_number": "label"}
    )
)
val_dataset = Dataset.from_pandas(
    val_df[["Review", "Label_number"]].rename(
        columns={"Review": "text", "Label_number": "label"}
    )
)

# -- Tokenize toàn bộ dataset --
print("\nĐang tokenize tập train...")
train_dataset = train_dataset.map(tokenize_function, batched=True, batch_size=1000)
print("Đang tokenize tập validation...")
val_dataset = val_dataset.map(tokenize_function, batched=True, batch_size=1000)

# Đặt format cho PyTorch
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print(f"\nTokenization hoàn tất!")
print(f"  Train dataset : {train_dataset}")
print(f"  Val dataset   : {val_dataset}")

In [ ]:
# =============================================================================
# Cell 9: Cấu hình mô hình và Training Arguments
# =============================================================================
# -- Tải mô hình PhoBERT với classification head (3 lớp) --
print("Đang tải mô hình PhoBERT-base-v2...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    # Gán nhãn cho mô hình (hữu ích khi inference)
    id2label=LABEL_MAP,
    label2id={v: k for k, v in LABEL_MAP.items()},
)
model.to(device)

# Đếm số lượng tham số
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Tổng số tham số     : {total_params:,}")
print(f"Tham số trainable   : {trainable_params:,}")

# -- Dynamic padding: pad theo batch thay vì pad toàn bộ về MAX_LENGTH --
# Giúp tiết kiệm memory GPU và tăng tốc training
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding="longest",          # Pad theo câu dài nhất trong batch
    pad_to_multiple_of=8,       # Tối ưu cho Tensor Core (GPU T4)
)

# -- Hàm tính metrics --
def compute_metrics(eval_pred):
    """Tính các chỉ số đánh giá: accuracy, precision, recall, f1."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    # Macro: tính trung bình không trọng số giữa các lớp
    prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    # Weighted: tính trung bình có trọng số theo số mẫu mỗi lớp
    prec_weighted, rec_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted", zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision_macro": prec_macro,
        "recall_macro": rec_macro,
        "f1_macro": f1_macro,
        "precision_weighted": prec_weighted,
        "recall_weighted": rec_weighted,
        "f1_weighted": f1_weighted,
    }

# -- Tính số bước training để cấu hình warmup --
BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2  # Effective batch size = 16 * 2 = 32
NUM_EPOCHS = 5

effective_batch = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
steps_per_epoch = len(train_dataset) // effective_batch
total_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = int(total_steps * 0.1)  # 10% bước đầu warm up

print(f"\n=== CẤU HÌNH TRAINING ===")
print(f"  Batch size (per device)    : {BATCH_SIZE}")
print(f"  Gradient accumulation      : {GRADIENT_ACCUMULATION_STEPS}")
print(f"  Effective batch size       : {effective_batch}")
print(f"  Epochs                     : {NUM_EPOCHS}")
print(f"  Steps per epoch            : {steps_per_epoch}")
print(f"  Total steps                : {total_steps}")
print(f"  Warmup steps               : {warmup_steps}")

# -- Training Arguments --
training_args = TrainingArguments(
    output_dir="./phobert-v2-training",       # Thư mục tạm (không lưu checkpoint lên Drive)
    overwrite_output_dir=True,

    # --- Số epoch và batch ---
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2, # Eval có thể dùng batch lớn hơn
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    # --- Learning rate và optimizer ---
    learning_rate=2e-5,
    weight_decay=0.01,
    adam_beta1=0.9,
    adam_beta2=0.999,
    adam_epsilon=1e-8,
    max_grad_norm=1.0,                        # Gradient clipping

    # --- Scheduler ---
    lr_scheduler_type="cosine",               # Cosine annealing
    warmup_steps=warmup_steps,

    # --- Mixed precision ---
    fp16=True,                                # Tối ưu cho GPU T4

    # --- Evaluation ---
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,

    # --- Lựa chọn mô hình tốt nhất ---
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    # --- Không lưu checkpoint lên Drive ---
    save_total_limit=1,

    # --- Logging ---
    report_to="none",                         # Không gửi lên wandb/tensorboard
    disable_tqdm=False,

    # --- Seed ---
    seed=SEED,
    data_seed=SEED,

    # --- Tối ưu memory ---
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    group_by_length=True,                     # Nhóm các câu có độ dài gần nhau
)

print("\nTraining Arguments đã sẵn sàng!")

In [ ]:
# =============================================================================
# Cell 10: Bắt đầu Fine-tune mô hình
# =============================================================================
# -- Tạo Trainer --
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2,        # Dừng sớm nếu 2 epoch không cải thiện
            early_stopping_threshold=0.001,
        ),
    ],
)

# -- Giải phóng VRAM trước khi train --
torch.cuda.empty_cache()

print("=== BẮT ĐẦU TRAINING ===")
print(f"Mô hình : {MODEL_NAME}")
print(f"Dataset : {len(train_dataset)} train / {len(val_dataset)} val")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print()

# -- Train --
train_result = trainer.train()

# -- In kết quả training --
print("\n=== KẾT QUẢ TRAINING ===")
print(f"  Training loss: {train_result.training_loss:.4f}")
print(f"  Thời gian    : {train_result.metrics.get('train_runtime', 0):.1f} giây")
print(f"  Samples/sec  : {train_result.metrics.get('train_samples_per_second', 0):.1f}")

In [ ]:
# =============================================================================
# Cell 11: Đánh giá mô hình trên tập validation
# =============================================================================
print("=== ĐÁNH GIÁ TRÊN TẬP VALIDATION ===\n")

# -- Dự đoán trên tập validation --
eval_result = trainer.evaluate()

# In các chỉ số chính
print(f"Accuracy           : {eval_result['eval_accuracy']:.4f}")
print(f"Precision (macro)  : {eval_result['eval_precision_macro']:.4f}")
print(f"Recall (macro)     : {eval_result['eval_recall_macro']:.4f}")
print(f"F1-score (macro)   : {eval_result['eval_f1_macro']:.4f}")
print(f"F1-score (weighted): {eval_result['eval_f1_weighted']:.4f}")

# -- Lấy dự đoán chi tiết để in classification report --
predictions_output = trainer.predict(val_dataset)
preds = np.argmax(predictions_output.predictions, axis=-1)
labels = predictions_output.label_ids

print(f"\n=== CLASSIFICATION REPORT ===")
print(classification_report(labels, preds, target_names=LABEL_NAMES, digits=4))

In [ ]:
# =============================================================================
# Cell 12: Visualization -- Training curves và Confusion Matrix
# =============================================================================
# -- Lấy lịch sử training từ log --
log_history = trainer.state.log_history

# Trích xuất training loss theo step
train_steps = []
train_losses = []
for entry in log_history:
    if "loss" in entry and "eval_loss" not in entry:
        train_steps.append(entry.get("step", 0))
        train_losses.append(entry["loss"])

# Trích xuất eval metrics theo epoch
eval_epochs = []
eval_accuracies = []
eval_f1s = []
eval_losses = []
for entry in log_history:
    if "eval_accuracy" in entry:
        eval_epochs.append(entry.get("epoch", 0))
        eval_accuracies.append(entry["eval_accuracy"])
        eval_f1s.append(entry.get("eval_f1_macro", 0))
        eval_losses.append(entry.get("eval_loss", 0))

# -- Vẽ biểu đồ --
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Biểu đồ 1: Training Loss
axes[0].plot(train_steps, train_losses, color="#e74c3c", linewidth=1.5)
axes[0].set_title("Training Loss", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

# Biểu đồ 2: Validation Accuracy và F1-score
if eval_epochs:
    axes[1].plot(eval_epochs, eval_accuracies, "o-", color="#2ecc71", label="Accuracy", linewidth=2)
    axes[1].plot(eval_epochs, eval_f1s, "s-", color="#3498db", label="F1 (macro)", linewidth=2)
    axes[1].set_title("Validation Metrics", fontsize=14, fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Score")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim([0.0, 1.05])

# Biểu đồ 3: Confusion Matrix
cm = confusion_matrix(labels, preds)
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=LABEL_NAMES,
    yticklabels=LABEL_NAMES,
    ax=axes[2],
    cbar_kws={"shrink": 0.8},
)
axes[2].set_title("Confusion Matrix", fontsize=14, fontweight="bold")
axes[2].set_xlabel("Dự đoán")
axes[2].set_ylabel("Thực tế")

plt.tight_layout()
plt.show()

# -- In confusion matrix dạng số --
print("=== CONFUSION MATRIX ===")
print(f"{'':>8} {'neg':>8} {'pos':>8} {'neu':>8}")
for i, name in enumerate(LABEL_NAMES):
    row = "  ".join(f"{cm[i][j]:>6}" for j in range(NUM_LABELS))
    print(f"{name:>8} {row}")

In [ ]:
# =============================================================================
# Cell 13: Lưu mô hình và tokenizer lên Google Drive
# =============================================================================
print(f"Đang lưu mô hình và tokenizer vào: {OUTPUT_DIR}")

# Chỉ lưu mô hình cuối cùng (best model) và tokenizer
# Không lưu checkpoint trong quá trình training
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Lưu kèm metrics để tham khảo sau
metrics_to_save = {
    "accuracy": eval_result["eval_accuracy"],
    "f1_macro": eval_result["eval_f1_macro"],
    "f1_weighted": eval_result["eval_f1_weighted"],
    "precision_macro": eval_result["eval_precision_macro"],
    "recall_macro": eval_result["eval_recall_macro"],
    "model_name": MODEL_NAME,
    "num_labels": NUM_LABELS,
    "max_length": MAX_LENGTH,
    "epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": 2e-5,
    "label_map": LABEL_MAP,
}

metrics_path = os.path.join(OUTPUT_DIR, "training_metrics.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics_to_save, f, ensure_ascii=False, indent=2)

# Kiểm tra các file đã lưu
saved_files = os.listdir(OUTPUT_DIR)
print(f"\nCác file đã lưu ({len(saved_files)} files):")
for fname in sorted(saved_files):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  {fname} ({size_mb:.1f} MB)")

print(f"\nLưu mô hình thành công!")

In [ ]:
# =============================================================================
# Cell 14: Inference demo -- Dự đoán cảm xúc từ câu tiếng Việt
# =============================================================================

def predict_sentiment(text, model, tokenizer, device):
    """Dự đoán cảm xúc cho một câu tiếng Việt.

    Args:
        text: Câu cần phân tích cảm xúc.
        model: Mô hình đã fine-tune.
        tokenizer: PhoBERT tokenizer.
        device: CPU hoặc GPU.

    Returns:
        Dict chứa nhãn dự đoán, độ tin cậy, và xác suất từng lớp.
    """
    model.eval()
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]

    pred_label = int(np.argmax(probs))
    pred_name = LABEL_MAP[pred_label]
    confidence = float(probs[pred_label])

    return {
        "text": text,
        "label": pred_name,
        "label_id": pred_label,
        "confidence": confidence,
        "probabilities": {LABEL_MAP[i]: float(probs[i]) for i in range(NUM_LABELS)},
    }


# -- Thử nghiệm với một số câu mẫu --
test_sentences = [
    "Sản phẩm rất tốt, chất lượng tuyệt vời, sẽ mua lại",
    "Hàng kém chất lượng, giao sai màu, thất vọng",
    "Hàng bình thường, không có gì đặc biệt",
    "Đội ngũ hỗ trợ nhiệt tình, giao hàng nhanh, đóng gói cẩn thận",
    "Mua về dùng 2 ngày đã hỏng, tiền nào của nấy",
]

print("=== INFERENCE DEMO ===\n")
for sentence in test_sentences:
    result = predict_sentiment(sentence, model, tokenizer, device)
    print(f"Câu    : {result['text']}")
    print(f"Kết quả: {result['label']} (độ tin cậy: {result['confidence']:.4f})")
    prob_str = " | ".join(
        f"{name}: {prob:.4f}" for name, prob in result["probabilities"].items()
    )
    print(f"Chi tiết: {prob_str}")
    print("-" * 60)

In [ ]:
# =============================================================================
# Cell 15: Tải lại mô hình đã lưu (để sử dụng sau này)
# =============================================================================
# Cell này minh họa cách tải lại mô hình từ Google Drive
# Không cần chạy lại training, chỉ cần chạy cell này

# SAVED_MODEL_DIR = "/content/drive/MyDrive/phobert-v2"
#
# loaded_tokenizer = AutoTokenizer.from_pretrained(SAVED_MODEL_DIR)
# loaded_model = AutoModelForSequenceClassification.from_pretrained(SAVED_MODEL_DIR)
# loaded_model.to(device)
# loaded_model.eval()
#
# result = predict_sentiment(
#     "Sản phẩm này rất đẹp, mình rất hài lòng",
#     loaded_model, loaded_tokenizer, device
# )
# print(result)

# -- Giải phóng GPU memory --
torch.cuda.empty_cache()
print("Hoàn tất. Mô hình đã được lưu tại:", OUTPUT_DIR)
print(f"GPU memory đã sử dụng: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
print(f"GPU memory đã cấp phát: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")